# 04 - Vision Transformer (from scratch) on PneumoniaMNIST

Second of the three architectures being compared. Implemented from scratch in `src/models.py` (`ViT`), including a manual multi-head self-attention (no `nn.MultiheadAttention`), so every step of the computation is explicit.

**Sizing note:** the original ViT (Dosovitskiy et al., 2020) uses 16x16 patches on 224x224 images, embed_dim=768, depth=12 — defaults meant for large-scale pretraining on millions of images. Applied directly to 28x28 images with ~4.7k training samples, that config would leave almost no patches per image and would overfit badly. This version uses 4x4 patches (49 patches per image), embed_dim=64, depth=4, 4 attention heads — a much smaller transformer, sized for this dataset instead of ImageNet.

**Hypothesis to test:** ViT lacks the inductive biases CNNs have (locality, translation invariance) and is known to need more data to compensate. With ~4.7k training images and no pretraining, we should expect ViT to underperform both SimpleCNN and ResNet-18.

In [ ]:
import sys
sys.path.append("..")

import torch

from src.data import get_dataloaders
from src.models import build_model
from src.train import train_model, evaluate_on_test
from src.utils import set_seed, get_device

set_seed(42)
device = get_device()
print("Using device:", device)


## 1. Load data

In [ ]:
FLAG = "pneumoniamnist"
SIZE = 28
BATCH_SIZE = 128

loaders, spec = get_dataloaders(FLAG, size=SIZE, batch_size=BATCH_SIZE,
                                root="../data", use_train_stats=True)
print(spec)

## 2. Build the ViT

In [ ]:
model = build_model("vit", n_channels=spec.n_channels, n_classes=spec.n_classes)
n_params = sum(p.numel() for p in model.parameters())
print(f"ViT has {n_params:,} parameters")
# For reference: SimpleCNN ~23.6k | ResNet-18 ~11.17M | ViT sits in between.

## 3. Train

In [ ]:
history = train_model(
    model, loaders, spec, device,
    epochs=15,
    lr=1e-3,
    out_dir="../results",
    run_name="vit_pneumoniamnist",
    monitor="auc",
    early_stopping_patience=5,
)

## 4. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

train_hist, val_hist = history["train"], history["val"]
epochs_ran = [h["epoch"] for h in train_hist]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_ran, [h["loss"] for h in train_hist], label="train")
axes[0].plot(epochs_ran, [h["loss"] for h in val_hist], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(epochs_ran, [h["auc"] for h in train_hist], label="train")
axes[1].plot(epochs_ran, [h["auc"] for h in val_hist], label="val")
axes[1].axhline(val_hist[0]["majority_baseline_accuracy"], color="gray",
                linestyle="--", label="majority baseline (accuracy)")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("AUC"); axes[1].legend()
axes[1].set_title("AUC")
plt.tight_layout(); plt.show()

## 5. Final evaluation on the test split

In [ ]:
test_result = evaluate_on_test(model, loaders, spec, device,
                               out_dir="../results",
                               run_name="vit_pneumoniamnist")
test_result

## 6. Confusion matrix

In [ ]:
cm = test_result.confusion
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")
labels = [spec.labels[str(c)] for c in range(spec.n_classes)]
ax.set_xticks(range(spec.n_classes)); ax.set_xticklabels(labels)
ax.set_yticks(range(spec.n_classes)); ax.set_yticklabels(labels)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Test set confusion matrix")
plt.tight_layout(); plt.show()

## 7. Compare against SimpleCNN and ResNet-18

This is the first three-way comparison, and the key test of this project's research question so far: does the lack of inductive bias hurt ViT as much as the literature predicts, in this low-data regime?

In [ ]:
import json
import pandas as pd

with open("../results/simple_cnn_pneumoniamnist_test_metrics.json") as f:
    cnn_metrics = json.load(f)
with open("../results/resnet18_pneumoniamnist_test_metrics.json") as f:
    resnet_metrics = json.load(f)

comparison = pd.DataFrame([
    {"model": "SimpleCNN", **cnn_metrics},
    {"model": "ResNet-18", **resnet_metrics},
    {"model": "ViT", **test_result.as_dict()},
]).set_index("model")
comparison[["accuracy", "balanced_accuracy", "auc", "f1", "sensitivity", "specificity"]]

## 8. Observations

_Write your conclusions here: did ViT underperform the two CNN-based models, as the "ViT needs more data" hypothesis predicts? Was the gap large or small? Look at the train/val loss gap — does it suggest ViT overfits more or less than the CNNs, given it lacks their built-in locality bias? This result will be the baseline for judging whether MedViT's hybrid design actually helps._